In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import torch
from tabpfn import TabPFNRegressor
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yaml
from src.real_data.quotes import load, clean, forwards, invert
from src.model.SSVI import fit_ssvi, ssvi_iv_pointwise
import warnings
warnings.filterwarnings("ignore", message="Running on CPU with more than")

In [ ]:
cfg = yaml.safe_load(open(REPO / "config.yaml"))
raw = load(REPO / "datasets/raw/spxw/2026-07-08.dbn.zst", "2026-07-08")
eod = clean(raw[raw.ts_recv == raw.ts_recv.max()])
q = invert(eod, forwards(eod))
q = q[(q.z >= -1.5) & (q.z <= 0.5) & (q.tau >= 0.02) & (q.tau <= 1.0)].reset_index(drop=True)

def sample_context(quotes, n, sigma=0.25, rng=None):
    rng = rng or np.random.default_rng()
    zw = np.exp(-0.5 * (quotes["z"] / sigma) ** 2)             # gaussian ATM in z, sigma matches synthetic
    w = zw / zw.groupby(quotes["expiry"]).transform("sum")    # equal weight per expiry ~ uniform in ttm
    idx = rng.choice(len(quotes), min(n, len(quotes)), replace=False, p=(w / w.sum()).to_numpy())
    return quotes.iloc[idx]

In [ ]:
Ns = [5, 10, 20, 40, 80, 160]
mae = {n: [] for n in Ns}
for n in Ns:
    for seed in range(15):
        ctx = sample_context(q, n, rng=np.random.default_rng(seed))
        held = q.drop(ctx.index)
        params, _ = fit_ssvi(ctx[["k", "tau"]].to_numpy(), ctx["mid_iv"].to_numpy(), cfg)
        pred = ssvi_iv_pointwise(held["tau"].to_numpy(), held["k"].to_numpy(), params)
        mae[n].append(np.mean(np.abs(pred - held["mid_iv"].to_numpy())))
for n in Ns:
    print(f"SSVI   N={n:3d}:  MAE {np.mean(mae[n]) * 100:.3f} ± {np.std(mae[n]) * 100:.3f} %")

In [ ]:
BID, ASK, TRUE = -1.0, 1.0, 0.0
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TabPFNRegressor(n_estimators=1, inference_config={"FINGERPRINT_FEATURE": False}, device=device)
state = torch.load(REPO / "checkpoints/supervised_z_full/final.pt", map_location=device)

mae_model = {n: [] for n in Ns}
for n in Ns:
    for seed in range(15):
        ctx = sample_context(q, n, rng=np.random.default_rng(seed))
        held = q.drop(ctx.index)
        X = np.column_stack([np.tile(ctx.z, 2), np.tile(ctx.tau, 2), np.repeat([BID, ASK], len(ctx))])
        model.fit(X, np.concatenate([ctx.bid_iv, ctx.ask_iv]))
        model.model_.load_state_dict(state)
        Xq = np.column_stack([held.z, held.tau, np.full(len(held), TRUE)])
        mae_model[n].append(np.mean(np.abs(model.predict(Xq) - held["mid_iv"].to_numpy())))
    print(f"N={n:3d}:  MAE {np.mean(mae_model[n])*100:.3f} ± {np.std(mae_model[n])*100:.3f} %")


for series, label, marker in [(mae, "SSVI", "o"), (mae_model, "supervised_z_full", "s")]:
    m = [np.mean(series[n]) * 100 for n in Ns]
    s = [np.std(series[n]) * 100 for n in Ns]
    plt.errorbar(Ns, m, yerr=s, marker=marker, capsize=3, label=label)
plt.xscale("log"); plt.xlabel("N context"); plt.ylabel("held-out MAE (%)")
plt.title("reconstruction vs context size"); plt.legend()

In [ ]:
ctx = sample_context(q, 40, rng=np.random.default_rng(0))
held = q.drop(ctx.index)

X = np.column_stack([np.tile(ctx.z, 2), np.tile(ctx.tau, 2), np.repeat([BID, ASK], len(ctx))])
model.fit(X, np.concatenate([ctx.bid_iv, ctx.ask_iv]))
model.model_.load_state_dict(state)
Xq = np.column_stack([held.z, held.tau, np.full(len(held), TRUE)])
held = held.assign(pred=model.predict(Xq))

params, _ = fit_ssvi(ctx[["k", "tau"]].to_numpy(), ctx["mid_iv"].to_numpy(), cfg)
held = held.assign(ssvi=ssvi_iv_pointwise(held["tau"].to_numpy(), held["k"].to_numpy(), params))
print(f"context {len(ctx)}   held-out {len(held)}   expiries {ctx.expiry.nunique()}/{q.expiry.nunique()}")

In [ ]:
dtes = sorted(held.dte.unique())
targets = sorted({min(dtes, key=lambda d: abs(d - x)) for x in [8, 30, 90, 250]})

fig, axes = plt.subplots(1, len(targets), figsize=(3.6 * len(targets), 3.4), sharey=False)
for ax, d in zip(np.atleast_1d(axes), targets):
    h = held[held.dte == d].sort_values("z")
    c = ctx[ctx.dte == d]
    ax.fill_between(h.z, h.bid_iv * 100, h.ask_iv * 100, color="0.8", label="bid/ask")
    ax.plot(h.z, h.ssvi * 100, color="C0", lw=1.5, label="SSVI")
    ax.plot(h.z, h.pred * 100, color="C3", lw=1.8, label="model")
    ax.plot(c.z, c.mid_iv * 100, "k*", ms=9, label="context")
    ax.set_title(f"{d}d", fontsize=11)
    ax.set_xlabel("z")
    ax.grid(alpha=0.25)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
np.atleast_1d(axes)[0].set_ylabel("IV (%)")
np.atleast_1d(axes)[-1].legend(frameon=False, fontsize=9)
fig.suptitle("model vs SSVI vs bid/ask — EOD 2026-07-08", y=1.02)
fig.tight_layout()

In [ ]:
def inside(p):
    return np.mean((p >= held.bid_iv.to_numpy()) & (p <= held.ask_iv.to_numpy()))

half = np.mean((held.ask_iv - held.bid_iv) / 2) * 100
print(f"held-out quotes: {len(held)}   avg half-spread: {half:.3f}% IV\n")
for name, p in [("model", held.pred.to_numpy()), ("SSVI", held.ssvi.to_numpy())]:
    mae = np.mean(np.abs(p - held.mid_iv.to_numpy())) * 100
    print(f"{name:6}  inside-spread {inside(p):6.1%}   MAE-vs-mid {mae:.3f}%")